# Eliminacion_Ruido

Este notebook elimina un grupo de variables candidatas a **ruido** del dataset final de entrenamiento.

La idea no es eliminar variables al azar, sino retirar columnas que muestran bajo aporte predictivo o poca utilidad práctica según tres señales principales:

1. **Dominancia alta:** casi todos los registros tienen el mismo valor, por lo que la variable separa poco a los clientes.
2. **Importancia por permutación cercana a cero o negativa:** al desordenar la variable, el modelo casi no pierde desempeño o incluso mejora ligeramente.
3. **Baja relación con `TARGET`:** correlación o señal estadística muy débil frente al objetivo.

> Importante: este notebook elimina solo el grupo de variables de baja señal/ruido. No elimina las variables redundantes del grupo de correlaciones 0.86–0.90 ni las variables borderline.

## 1. Carga de librerías y datos

Se carga el dataset final ya unido (`final_train_features.parquet`). Este archivo debe contener `SK_ID_CURR`, `TARGET` y las variables finales candidatas.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

# Base del proyecto (asume ejecución desde la carpeta del notebook)
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'outputs'


def find_first_existing(candidates):
    """Devuelve la primera ruta existente de una lista de candidatos."""
    for p in candidates:
        if p.exists():
            return p
    return None


# Intentos ordenados: primero dataset ya seleccionado, luego dataset amplio
possible_train_paths = [
    DATA_DIR / 'train_selected_features.parquet',
    PROJECT_DIR / 'train_selected_features.parquet',
    DATA_DIR / 'final_train_features.parquet',
    PROJECT_DIR / 'final_train_features.parquet',
]

# Búsqueda adicional por si el archivo está en subcarpetas del proyecto
possible_train_paths.extend(PROJECT_DIR.glob('**/train_selected_features.parquet'))
possible_train_paths.extend(PROJECT_DIR.glob('**/final_train_features.parquet'))

train_path = find_first_existing(possible_train_paths)

if train_path is None:
    raise FileNotFoundError(
        "No se encontró ni 'train_selected_features.parquet' ni "
        "'final_train_features.parquet'. "
        f"Ubícalos en '{PROJECT_DIR}' o en '{DATA_DIR}'."
    )

final_train = pd.read_parquet(train_path)

print('Proyecto:', PROJECT_DIR)
print('Archivo cargado:', train_path)
print('Shape inicial:', final_train.shape)
print('Tiene SK_ID_CURR:', 'SK_ID_CURR' in final_train.columns)
print('Tiene TARGET:', 'TARGET' in final_train.columns)
final_train.head()

Proyecto: C:\Users\camil\Music\PI\Proyecto_Integrador
Archivo cargado: C:\Users\camil\Music\PI\Proyecto_Integrador\data\train_selected_features.parquet
Shape inicial: (307511, 158)
Tiene SK_ID_CURR: True
Tiene TARGET: True


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,BASEMENTAREA_AVG,YEARS_BUILD_AVG,FLOORSMAX_AVG,LANDAREA_AVG,NONLIVINGAREA_AVG,FONDKAPREMONT_MODE,HOUSETYPE_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,FLAG_DOCUMENT_16,FLAG_DOCUMENT_18,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AGE_YEARS,YEARS_EMPLOYED,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_GOODS_RATIO,INCOME_PER_PERSON,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,BUREAU_QUERIES_TOTAL,bureau_n_active,bureau_n_closed,bureau_n_bad,bureau_avg_days_credit,bureau_min_days_credit,bureau_avg_days_update,bureau_total_credit,bureau_total_debt,bureau_total_overdue_amt,bureau_n_prolonged,bureau_n_card,bureau_n_mortgage,bureau_n_car_loan,bureau_active_pct,bureau_closed_pct,bureau_debt_ratio,bureau_mean_credit_duration,bureau_mean_days_overrun,bureau_pct_paid_early,no_bureau_history,bb_months_count_sum,bb_months_span_mean,bb_n_mora_sum,bb_has_mora_rate,bb_pct_mora_mean,bb_pct_mora_max,bb_pct_X_mean,bb_max_severity_max,no_bureau_balance_history,prev_app_count,prev_approved_rate,prev_refused_rate,prev_canceled_rate,prev_unused_offer_rate,prev_contract_type_cash_rate,prev_contract_type_consumer_rate,prev_contract_type_revolving_rate,prev_annuity_approved_mean,prev_annuity_approved_max,prev_credit_approved_mean,prev_credit_application_ratio_mean,prev_goods_price_mean,prev_goods_price_max,prev_cnt_payment_mean,prev_cnt_payment_max,prev_reject_hc_rate,prev_reject_limit_rate,prev_reject_sco_rate,prev_yield_high_rate,prev_yield_low_rate,prev_insured_on_approval_rate,prev_insured_on_approval_count,inst_prev_credit_count,inst_total_installments_count,inst_late_payment_rate,inst_days_past_due_mean,inst_days_past_due_max,inst_on_time_payment_rate,inst_underpayment_rate,inst_overpayment_rate,inst_total_payment_ratio,inst_amt_payment_mean,inst_calendar_version_max,inst_credit_card_rate,no_installments_history,pos_cash_active_rate,pos_cash_completed_rate,pos_cash_dpd_def_positive_rate,pos_cash_dpd_def_max,pos_cash_recent_active_count,pos_cash_recent_installments_future_mean,pos_cash_recent_installments_future_max,cc_months_count,cc_active_rate,cc_completed_rate,cc_dpd_positive_rate,cc_dpd_def_positive_rate,cc_dpd_max,cc_amt_balance_mean,cc_credit_limit_mean,cc_utilization_mean,cc_utilization_max,cc_payment_current_max,cc_payment_total_mean,cc_payment_total_sum,cc_payment_min_ratio_mean,cc_drawings_current_mean,cc_drawings_atm_current_mean,cc_drawings_pos_current_mean,cc_drawings_positive_rate,cc_installments_mature_cum_max
0,100002,1,Cash loans,M,N,Y,0,202500.0,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-3648.0,-2120,0.0,1,0,1,0,Laborers,1.0,2,WEDNESDAY,10,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0369,0.6192,0.0833,0.0369,0.0000,reg oper account,block of flats,"Stone, brick",No,2.0,2.0,2.0,-1134.0,1,0,0,0,0,0,0.0,0.0,25.902806,1.744011,2.007889,0.121978,1.158397,202500.0,0.161787,0.083037,1.0,2.0,6.0,0.0,-874.00,-1437.0,-499.875,865055.565,245781.0,0.0,0.0,4.0,0.0,0.0,0.25,0.75,0.284121,277.000000,-163.5,0.166667,0,110.0,12.75,27.0,0.75,0.255682,0.5,0.161932,1.0,0,1.0,1.000000,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,9251.775,9251.775,179055.00,1.000000,1790

## 2. Variables candidatas a eliminar por ruido

Estas variables fueron marcadas como candidatas porque tienen baja señal predictiva, alta dominancia, permutación cercana a cero/negativa o interpretación débil frente al impago.

| Variable | Información que da | Razón de eliminación |
|---|---|---|
| `FLAG_DOCUMENT_18` | Indica si el cliente presentó el documento 18. | Casi todos los clientes tienen el mismo valor y la permutación es 0. |
| `FLAG_DOCUMENT_5` | Indica si presentó el documento 5. | Muy poca variación y aporte casi nulo. |
| `REG_REGION_NOT_LIVE_REGION` | Indica si la región registrada difiere de la región donde vive. | Casi todos tienen el mismo valor y la señal frente al target es muy baja. |
| `bureau_n_prolonged` | Número de créditos prolongados en buró. | Casi todos tienen 0; bajo aporte marginal. |
| `prev_unused_offer_rate` | Proporción de ofertas previas aprobadas pero no usadas. | Baja señal y permutación negativa. |
| `FLAG_EMAIL` | Indica si el cliente registró email. | Alta dominancia y aporte predictivo muy bajo. |
| `cc_dpd_def_positive_rate` | Proporción de meses de tarjeta con mora definida positiva. | Señal débil adicional frente a otras variables de mora. |
| `FLAG_DOCUMENT_8` | Indica si presentó el documento 8. | Alta dominancia y permutación negativa. |
| `no_bureau_balance_history` | Indica ausencia de historial en `bureau_balance`. | Redundante con `no_bureau_history` y bajo aporte marginal. |
| `bb_pct_X_mean` | Promedio de meses con estado `X` en `bureau_balance`. | Señal débil y permutación negativa. |
| `NAME_TYPE_SUITE` | Quién acompañó al cliente al solicitar el crédito. | Baja importancia; variable débil para predecir impago. |
| `FLAG_OWN_REALTY` | Indica si el cliente tiene propiedad/vivienda. | Baja señal frente al target. |
| `WEEKDAY_APPR_PROCESS_START` | Día de la semana de inicio de la solicitud. | Señal débil y demasiado ruidosa para predicción. |

In [3]:
features_ruido = [
    'FLAG_DOCUMENT_18',
    'FLAG_DOCUMENT_5',
    'REG_REGION_NOT_LIVE_REGION',
    'bureau_n_prolonged',
    'prev_unused_offer_rate',
    'FLAG_EMAIL',
    'cc_dpd_def_positive_rate',
    'FLAG_DOCUMENT_8',
    'no_bureau_balance_history',
    'bb_pct_X_mean',
    'NAME_TYPE_SUITE',
    'FLAG_OWN_REALTY',
    'WEEKDAY_APPR_PROCESS_START'
]

razones_ruido = {
    'FLAG_DOCUMENT_18': 'Documento con muy poca variación; permutación nula.',
    'FLAG_DOCUMENT_5': 'Documento con alta dominancia; aporte casi cero.',
    'REG_REGION_NOT_LIVE_REGION': 'Variable geográfica casi constante y con señal muy débil.',
    'bureau_n_prolonged': 'Casi todos los clientes tienen 0 créditos prolongados; baja señal.',
    'prev_unused_offer_rate': 'Baja señal predictiva y permutación negativa.',
    'FLAG_EMAIL': 'Alta dominancia y aporte marginal muy bajo.',
    'cc_dpd_def_positive_rate': 'Baja señal adicional frente a otras variables de mora.',
    'FLAG_DOCUMENT_8': 'Alta dominancia y permutación negativa.',
    'no_bureau_balance_history': 'Redundante con no_bureau_history; bajo aporte marginal.',
    'bb_pct_X_mean': 'Variable de estado X en bureau_balance con señal débil.',
    'NAME_TYPE_SUITE': 'Quién acompañó al cliente no aporta señal fuerte de impago.',
    'FLAG_OWN_REALTY': 'Propiedad/vivienda con baja señal frente al target.',
    'WEEKDAY_APPR_PROCESS_START': 'Día de solicitud con baja señal predictiva.'
}

existencia = pd.DataFrame({
    'variable': features_ruido,
    'existe_en_dataset': [col in final_train.columns for col in features_ruido],
    'razon_eliminacion': [razones_ruido[col] for col in features_ruido]
})

existencia

,variable,existe_en_dataset,razon_eliminacion
0,FLAG_DOCUMENT_18,True,Documento con muy poca variación; permutación ...
1,FLAG_DOCUMENT_5,True,Documento con alta dominancia; aporte casi cero.
2,REG_REGION_NOT_LIVE_REGION,True,Variable geográfica casi constante y con señal...
3,bureau_n_prolonged,True,Casi todos los clientes tienen 0 créditos prol...
4,prev_unused_offer_rate,True,Baja señal predictiva y permutación negativa.
5,FLAG_EMAIL,True,Alta dominancia y aporte marginal muy bajo.
6,cc_dpd_def_positive_rate,True,Baja señal adicional frente a otras variables ...
7,FLAG_DOCUMENT_8,True,Alta dominancia y permutación negativa.
8,no_bureau_balance_history,True,Redundante con no_bureau_history; bajo aporte ...
9,bb_pct_X_mean,True,Variable de estado X en bureau_balance con señ...


## 3. Verificación de dominancia

La dominancia indica qué porcentaje de registros comparten el valor más frecuente de una variable.

Ejemplo: si `FLAG_EMAIL` tiene 94% de dominancia, significa que el 94% de los clientes tienen el mismo valor. Una variable así separa poco a los clientes, salvo que el grupo minoritario tenga una relación muy fuerte con `TARGET`.

In [4]:
dominancia_rows = []

for col in features_ruido:
    if col in final_train.columns:
        conteo = final_train[col].value_counts(dropna=False)
        conteo_pct = final_train[col].value_counts(normalize=True, dropna=False) * 100
        valor_dominante = conteo.index[0]
        cantidad_dominante = conteo.iloc[0]
        porcentaje_dominante = conteo_pct.iloc[0]
        n_categorias = final_train[col].nunique(dropna=False)
        dominancia_rows.append({
            'variable': col,
            'valor_dominante': valor_dominante,
            'cantidad_valor_dominante': cantidad_dominante,
            'porcentaje_dominancia': round(porcentaje_dominante, 2),
            'n_valores_unicos': n_categorias
        })

tabla_dominancia = pd.DataFrame(dominancia_rows).sort_values(
    'porcentaje_dominancia', ascending=False
).reset_index(drop=True)

tabla_dominancia

,variable,valor_dominante,cantidad_valor_dominante,porcentaje_dominancia,n_valores_unicos
0,FLAG_DOCUMENT_18,0,305011,99.19,2
1,FLAG_DOCUMENT_5,0,302863,98.49,2
2,REG_REGION_NOT_LIVE_REGION,0,302854,98.49,2
3,bureau_n_prolonged,0.0,299003,97.23,10
4,cc_dpd_def_positive_rate,0.0,292669,95.17,953
5,FLAG_EMAIL,0,290069,94.33,2
6,prev_unused_offer_rate,0.0,288478,93.81,117
7,FLAG_DOCUMENT_8,0,282487,91.86,2
8,NAME_TYPE_SUITE,Unaccompanied,249818,81.24,7
9,bb_pct_X_mean,0.0,234041,76.11,41965


## 4. Revisión opcional del reporte de selección de features

Si el archivo `08_final_feature_selection_report(1).csv` está disponible, se cargan las métricas técnicas: correlación con `TARGET`, información mutua, importancia por permutación y score combinado.

Estas métricas sirven para respaldar la eliminación.

In [5]:
possible_report_paths = [
    PROJECT_DIR / '08_final_feature_selection_report(1).csv',
    PROJECT_DIR / '08_final_feature_selection_report.csv',
    DATA_DIR / '08_final_feature_selection_report(1).csv',
    DATA_DIR / '08_final_feature_selection_report.csv',
]

# Búsqueda adicional del reporte en subcarpetas del proyecto
possible_report_paths.extend(PROJECT_DIR.glob('**/08_final_feature_selection_report(1).csv'))
possible_report_paths.extend(PROJECT_DIR.glob('**/08_final_feature_selection_report.csv'))

report_path = find_first_existing(possible_report_paths)

if report_path is not None:
    reporte = pd.read_csv(report_path)
    print('Reporte cargado:', report_path)
    cols_reporte = [
        col for col in [
            'column', 'selected', 'reason',
            'abs_pearson_corr_target', 'abs_spearman_corr_target',
            'mutual_info_target', 'perm_importance_mean', 'combined_score'
        ]
        if col in reporte.columns
    ]
    reporte_ruido = reporte[reporte['column'].isin(features_ruido)][cols_reporte].copy()
    display(reporte_ruido)
else:
    print(
        'No se encontró el reporte de selección en el proyecto. '
        'Se continúa solo con el dataset final.'
    )

Reporte cargado: C:\Users\camil\Music\PI\Proyecto_Integrador\data\feature_selection_reports\08_final_feature_selection_report.csv


,column,selected,reason,abs_pearson_corr_target,abs_spearman_corr_target,mutual_info_target,perm_importance_mean,combined_score
142,FLAG_DOCUMENT_8,True,correlación Pearson con TARGET; correlación Sp...,0.008592,0.008592,3.795561e-05,-0.000001,0.826733
143,FLAG_DOCUMENT_18,True,correlación Pearson con TARGET; correlación Sp...,0.007245,0.007245,2.872597e-05,0.000000,0.821782
145,bb_pct_X_mean,True,información mutua,0.000941,0.002178,4.835883e-04,-0.000008,0.752475
146,REG_REGION_NOT_LIVE_REGION,True,correlación Pearson con TARGET; correlación Sp...,0.004748,0.004748,1.086221e-05,-0.000001,0.678218
147,no_bureau_balance_history,True,importancia por permutación,0.001173,0.001173,6.874849e-07,0.000002,0.663366
148,bureau_n_prolonged,True,correlación Pearson con TARGET; correlación Sp...,0.005053,0.005053,1.240406e-05,-0.000003,0.653465
149,WEEKDAY_APPR_PROCESS_START,True,importancia por permutación,0.000000,0.000000,2.680248e-05,0.000021,0.638614
150,FLAG_EMAIL,True,importancia por permutación,0.000563,0.000563,1.591076e-07,0.000002,0.613861
151,NAME_TYPE_SUITE,True,importancia por permutación,0.000000,0.000000,5.786592e-05,0.000006,0.584158
152,FLAG_DOCUMENT_5,True,importancia por permutación,0.000385,0.000385,7.448059e-08,0.000002,0.579208


## 5. Eliminación de variables de ruido

Se eliminan únicamente las variables candidatas que realmente existen en el dataset.

No se eliminan `SK_ID_CURR` ni `TARGET`, porque se necesitan para identificación y entrenamiento/evaluación.

In [6]:
cols_eliminar = [col for col in features_ruido if col in final_train.columns]
cols_no_encontradas = [col for col in features_ruido if col not in final_train.columns]

print('Columnas candidatas:', len(features_ruido))
print('Columnas encontradas para eliminar:', len(cols_eliminar))
print('Columnas no encontradas:', len(cols_no_encontradas))
print('Shape antes:', final_train.shape)

final_train_sin_ruido = final_train.drop(columns=cols_eliminar).copy()

print('Shape después:', final_train_sin_ruido.shape)
print('Columnas eliminadas:', final_train.shape[1] - final_train_sin_ruido.shape[1])

Columnas candidatas: 13
Columnas encontradas para eliminar: 13
Columnas no encontradas: 0
Shape antes: (307511, 158)
Shape después: (307511, 145)
Columnas eliminadas: 13


In [7]:
tabla_eliminadas = pd.DataFrame({
    'variable_eliminada': cols_eliminar,
    'razon': [razones_ruido[col] for col in cols_eliminar]
})

tabla_eliminadas

,variable_eliminada,razon
0,FLAG_DOCUMENT_18,Documento con muy poca variación; permutación ...
1,FLAG_DOCUMENT_5,Documento con alta dominancia; aporte casi cero.
2,REG_REGION_NOT_LIVE_REGION,Variable geográfica casi constante y con señal...
3,bureau_n_prolonged,Casi todos los clientes tienen 0 créditos prol...
4,prev_unused_offer_rate,Baja señal predictiva y permutación negativa.
5,FLAG_EMAIL,Alta dominancia y aporte marginal muy bajo.
6,cc_dpd_def_positive_rate,Baja señal adicional frente a otras variables ...
7,FLAG_DOCUMENT_8,Alta dominancia y permutación negativa.
8,no_bureau_balance_history,Redundante con no_bureau_history; bajo aporte ...
9,bb_pct_X_mean,Variable de estado X en bureau_balance con señ...


## 6. Verificaciones después de eliminar

Se valida que:

1. Las variables eliminadas ya no estén en el dataset.
2. `SK_ID_CURR` y `TARGET` sigan presentes.
3. No se hayan eliminado filas.
4. Se conserve una cantidad de columnas consistente con la poda aplicada.

In [8]:
# 1. Variables eliminadas que todavía aparecen
variables_aun_presentes = [col for col in cols_eliminar if col in final_train_sin_ruido.columns]

# 2. Validar columnas clave
columnas_clave = ['SK_ID_CURR', 'TARGET']
columnas_clave_presentes = {col: col in final_train_sin_ruido.columns for col in columnas_clave}

# 3. Validar número de filas
filas_iguales = len(final_train_sin_ruido) == len(final_train)

print('Variables eliminadas aún presentes:', variables_aun_presentes)
print('Columnas clave presentes:', columnas_clave_presentes)
print('Filas iguales antes/después:', filas_iguales)
print('Filas antes:', len(final_train))
print('Filas después:', len(final_train_sin_ruido))
print('Columnas antes:', final_train.shape[1])
print('Columnas después:', final_train_sin_ruido.shape[1])

Variables eliminadas aún presentes: []
Columnas clave presentes: {'SK_ID_CURR': True, 'TARGET': True}
Filas iguales antes/después: True
Filas antes: 307511
Filas después: 307511
Columnas antes: 158
Columnas después: 145


## 7. Exportar dataset sin variables de ruido

Se exporta el dataset resultante en formato Parquet. También se guarda un CSV con el listado de variables eliminadas y su justificación.

In [ ]:
# Se centralizan salidas en la carpeta outputs del proyecto
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_dataset = OUTPUT_DIR / 'final_train_features_limpio.parquet'
output_report = OUTPUT_DIR / 'variables_eliminadas_ruido.csv'

# Exportar la tabla final del flujo (ruido + redundancia)
if 'final_train_limpio' in globals():
    final_train_limpio.to_parquet(output_dataset, index=False)
else:
    raise NameError(
        "No existe 'final_train_limpio'. Ejecuta primero la celda de segunda poda "
        "(variables redundantes) antes de exportar."
    )

tabla_eliminadas.to_csv(output_report, index=False)

print('Dataset final exportado:', output_dataset.resolve())
print('Reporte exportado:', output_report.resolve())

## Eliminación de variables por redundancia moderada

Después de la selección inicial de variables, se identificaron algunos pares de columnas con una correlación alta entre sí, aunque no alcanzaban el umbral original de eliminación automática definido en `0.90`. Estos pares presentaban correlaciones entre `0.86` y `0.90`, lo que indica que ambas variables dentro de cada par tienden a moverse de forma muy similar.

Aunque estas variables no son necesariamente incorrectas ni inútiles, mantener ambas puede introducir redundancia en el modelo. Es decir, el modelo recibe dos variables que explican información muy parecida. Para simplificar el conjunto de datos y reducir ruido, se decidió conservar la variable con mayor evidencia predictiva frente al `TARGET`, medida principalmente mediante el `combined_score`.

El `combined_score` resume distintas métricas de importancia, como correlación con el objetivo, información mutua e importancia por permutación. Sin embargo, la decisión no se basó únicamente en el score, sino también en la interpretación de negocio de cada variable.

| Variable eliminada | Variable conservada | Correlación | Justificación técnica | Justificación de negocio |
|---|---|---:|---|---|
| `inst_credit_card_rate` | `cc_utilization_max` | 0.897 | Ambas variables están altamente correlacionadas. Se conserva `cc_utilization_max` por tener mayor score combinado. | `cc_utilization_max` mide directamente el nivel máximo de uso del cupo de tarjeta de crédito, por lo que representa de forma más clara el comportamiento financiero del cliente en tarjetas. |
| `cc_payment_total_mean` | `cc_drawings_current_mean` | 0.895 | Las dos variables se mueven de forma muy similar. Se conserva la de mayor score. | El uso promedio de la tarjeta (`cc_drawings_current_mean`) puede explicar mejor el comportamiento de consumo. Quien usa más la tarjeta normalmente también realiza mayores pagos, por eso el pago promedio se vuelve redundante. |
| `NONLIVINGAREA_AVG` | `FLOORSMAX_AVG` | 0.890 | Variables del inmueble con alta correlación. Se conserva la de mayor score. | Ambas describen características físicas de la vivienda o edificio. `FLOORSMAX_AVG` puede capturar mejor el tipo de inmueble o nivel socioeconómico asociado al lugar de residencia. |
| `CNT_FAM_MEMBERS` | `CNT_CHILDREN` | 0.879 | Alta correlación entre tamaño familiar y número de hijos. Se conserva la variable con mejor score. | El número de hijos puede representar de forma más directa la carga económica familiar. Aunque `CNT_FAM_MEMBERS` también es relevante, aporta información parecida. |
| `bureau_min_days_credit` | `bureau_avg_days_credit` | 0.873 | Ambas resumen antigüedad del historial crediticio. Se conserva la de mayor score. | `bureau_avg_days_credit` resume la antigüedad promedio de todos los créditos del cliente, mientras que `bureau_min_days_credit` solo toma el crédito más antiguo. El promedio ofrece una visión más estable del historial. |
| `prev_cnt_payment_max` | `prev_cnt_payment_mean` | 0.870 | Alta correlación entre plazo máximo y plazo promedio de créditos previos. | El plazo promedio (`prev_cnt_payment_mean`) representa mejor el comportamiento general histórico del cliente, mientras que el máximo puede depender de un solo crédito extremo. |
| `prev_goods_price_mean` | `prev_goods_price_max` | 0.869 | Ambas resumen el valor de bienes financiados previamente. Se conserva la variable con mayor score. | Si el cliente ha financiado bienes costosos, tanto el promedio como el máximo tienden a crecer. Se mantiene el máximo porque puede capturar la mayor capacidad histórica de financiación observada. |
| `inst_on_time_payment_rate` | `inst_total_payment_ratio` | 0.866 | Ambas capturan calidad de pago. Se conserva la de mayor score. | `inst_total_payment_ratio` mide cuánto pagó el cliente frente a lo esperado, por lo que captura no solo si pagó a tiempo, sino también si pagó completo, por debajo o por encima de lo requerido. |
| `prev_annuity_approved_mean` | `prev_annuity_approved_max` | 0.864 | Alta correlación entre cuota promedio y cuota máxima aprobada. | La cuota máxima aprobada puede capturar la mayor capacidad de pago reconocida al cliente en solicitudes previas. Por eso se conserva frente al promedio. |
| `DEF_30_CNT_SOCIAL_CIRCLE` | `DEF_60_CNT_SOCIAL_CIRCLE` | 0.861 | Ambas variables miden riesgo del círculo social. Se conserva la de mayor score. | `DEF_60_CNT_SOCIAL_CIRCLE` representa un incumplimiento más severo que `DEF_30_CNT_SOCIAL_CIRCLE`, por lo que puede ser una señal más fuerte del riesgo del entorno social del cliente. |
| `INCOME_PER_PERSON` | `AMT_INCOME_TOTAL` | 0.862 | `INCOME_PER_PERSON` deriva directamente del ingreso total y el tamaño familiar. Se conserva la variable con mayor score. | `AMT_INCOME_TOTAL` representa directamente la capacidad económica declarada del cliente. Como `INCOME_PER_PERSON` depende de esta variable, puede introducir información redundante. |

### Criterio de decisión

La eliminación se realizó bajo la siguiente lógica:

1. **Correlación alta entre variables:**  
   Si dos variables tienen una correlación cercana a `0.90`, significa que contienen información muy parecida.

2. **Comparación del aporte frente al `TARGET`:**  
   Entre cada par, se conserva la variable con mayor `combined_score`, ya que tiene mayor evidencia de utilidad predictiva.

3. **Interpretación de negocio:**  
   Se revisa que la variable conservada tenga sentido desde el contexto financiero o crediticio. No se elimina únicamente por cálculo estadístico, sino porque la variable restante representa mejor el fenómeno que se quiere modelar.



In [10]:
# Segunda poda: variables redundantes adicionales
features_redundantes_eliminar = [
    "inst_credit_card_rate",
    "cc_payment_total_mean",
    "NONLIVINGAREA_AVG",
    "CNT_FAM_MEMBERS",
    "bureau_min_days_credit",
    "prev_cnt_payment_max",
    "prev_goods_price_mean",
    "inst_on_time_payment_rate",
    "prev_annuity_approved_mean",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "INCOME_PER_PERSON"
]

# Continuidad: seguir limpiando el resultado anterior
base_df = final_train_sin_ruido.copy()

features_existentes = [
    col for col in features_redundantes_eliminar
    if col in base_df.columns
]

features_no_existentes = [
    col for col in features_redundantes_eliminar
    if col not in base_df.columns
]

print("Columnas redundantes encontradas para eliminar:", len(features_existentes))
print(features_existentes)

print("\nColumnas redundantes no encontradas:", len(features_no_existentes))
print(features_no_existentes)

print("\nShape antes (sin ruido):", base_df.shape)

# Resultado final tras ruido + redundancia
final_train_limpio = base_df.drop(columns=features_existentes)

print("Shape después (sin ruido + sin redundantes):", final_train_limpio.shape)
print("Columnas eliminadas en esta etapa:", len(features_existentes))

# Exportación final en parquet
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_final_dataset = OUTPUT_DIR / 'final_train_features_limpio.parquet'
final_train_limpio.to_parquet(output_final_dataset, index=False)

print("\nDataset final exportado:", output_final_dataset.resolve())

Columnas redundantes encontradas para eliminar: 11
['inst_credit_card_rate', 'cc_payment_total_mean', 'NONLIVINGAREA_AVG', 'CNT_FAM_MEMBERS', 'bureau_min_days_credit', 'prev_cnt_payment_max', 'prev_goods_price_mean', 'inst_on_time_payment_rate', 'prev_annuity_approved_mean', 'DEF_30_CNT_SOCIAL_CIRCLE', 'INCOME_PER_PERSON']

Columnas redundantes no encontradas: 0
[]

Shape antes (sin ruido): (307511, 145)
Shape después (sin ruido + sin redundantes): (307511, 134)
Columnas eliminadas en esta etapa: 11

Dataset final exportado: C:\Users\camil\Music\PI\Proyecto_Integrador\outputs\final_train_features_limpio.parquet


## 8. Conclusión

Se eliminaron variables de baja señal predictiva y bajo aporte marginal. La decisión se justifica porque estas variables presentan una o varias de las siguientes condiciones:

- alta dominancia de un valor;
- importancia por permutación cercana a cero o negativa;
- baja relación con `TARGET`;
- posible redundancia con variables más informativas;
- interpretación de negocio débil para predecir impago.

Esta poda busca simplificar el dataset, reducir ruido y dejar un conjunto de variables más limpio para entrenar modelos. La validación final debe hacerse comparando métricas entre el modelo con todas las features seleccionadas y el modelo sin estas variables de ruido.